# 5. ExceedGAN for bivariate extremes

This notebook introduces **Fixed-Level ExceedGAN (FL-ExceedGAN)** and compares it with a standard GAN and EV-GAN.

For a stable workshop demonstration we use

$$
d=2,\qquad \mu=2,\qquad \gamma=0.5,\qquad
(\rho_1,\rho_2)=(-3,-3),
$$

with a Gumbel copula, Burr margins and

$$
\boldsymbol{\delta}_n=(0.1,0.1)^\top.
$$

Using $\rho_1=\rho_2=-3$ places both margins in the smoother second-order regime used in the EV-GAN approximation argument. The asymmetric setting from the slides,

$$
(\rho_1,\rho_2)=(-1,-3),
$$

is retained as a workshop extension.

The target is the upper-orthant exceedance distribution

$$
Y(\boldsymbol{\delta}_n)
\overset{d}{=}
X\mid
X^{(1)}>u_n^{(1)},\;
X^{(2)}>u_n^{(2)},
$$

where

$$
u_n^{(m)}=F_{X^{(m)}}^{-1}(1-\delta_n^{(m)}).
$$

Compared with the earlier version, this notebook also uses larger networks, a larger training sample, transformed discriminator inputs, repeated seeds and validation-based checkpoint selection.

## Learning objectives

By the end you should be able to:

1. construct $\mathcal Q(\boldsymbol\delta_n)$;
2. explain why exceedance generation is an extrapolation problem;
3. describe the log-spacing function;
4. explain the role of eLU activations;
5. construct GAN, EV-GAN and FL-ExceedGAN generators in PyTorch;
6. compare their marginal tail fit and bivariate dependence.

## Packages

If necessary:

```python
# %pip install torch numpy pandas matplotlib scipy statsmodels
```

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from scipy import stats as st
from scipy.special import expi
from scipy.stats import kendalltau
from statsmodels.distributions.copula.api import CopulaDistribution, GumbelCopula

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", device)

# 1. Simulate Gumbel-copula data with Burr margins

For margin $m$, use a Burr XII distribution with

$$
c_m=-\frac{\rho_m}{\gamma},
\qquad
d_m=-\frac{1}{\rho_m}.
$$

The Gumbel copula parameter is $\mu=2$.

For the default workshop run we set both second-order parameters to $-3$. This makes the EV-GAN baseline more stable and lets the comparison focus on the additional benefit of modelling the exceedance distribution directly.

In [ ]:
N_TRAIN = 100_000
N_TEST = 200_000

MU = 2.0
GAMMA = 0.5
RHOS = (-3.0, -3.0)
DELTA = np.array([0.10, 0.10], dtype=np.float32)

def generate_gumbel_burr(n, mu=MU, gamma=GAMMA, rhos=RHOS, seed=123):
    marginals = [
        st.burr12(c=-rho/gamma, d=-1.0/rho)
        for rho in rhos
    ]
    distribution = CopulaDistribution(
        copula=GumbelCopula(theta=mu, k_dim=2),
        marginals=marginals
    )
    return np.asarray(
        distribution.rvs(n, random_state=seed),
        dtype=np.float32
    )

X_train_full = generate_gumbel_burr(N_TRAIN, seed=SEED)
X_test_full = generate_gumbel_burr(N_TEST, seed=SEED + 1)

print("Training:", X_train_full.shape)
print("Test:", X_test_full.shape)

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X_train_full[:,0], X_train_full[:,1], s=6, alpha=0.2)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$X^{(1)}$")
plt.ylabel(r"$X^{(2)}$")
plt.title("Gumbel copula with Burr margins")
plt.show()

# 2. The upper-quadrant region

The slides define

$$
\mathcal Q(\boldsymbol\delta_n)
=
\left\{
x\in\mathbb R^2:
x^{(m)}>F_{X^{(m)}}^{-1}(1-\delta_n^{(m)}),
\;m=1,2
\right\}.
$$

We estimate the anchor points from the training sample and use those same anchors for the independent test set.

In [ ]:
def get_upper_quadrant(X, delta, anchor_points=None):
    X = np.asarray(X)
    delta = np.asarray(delta)

    if anchor_points is None:
        anchor_points = np.array([
            np.quantile(X[:, m], 1.0-delta[m])
            for m in range(X.shape[1])
        ])

    anchor_points = np.asarray(anchor_points, dtype=np.float32)

    mask = np.all(
        X > anchor_points.reshape(1, -1),
        axis=1
    )

    return X[mask], anchor_points

X_train, anchor_points = get_upper_quadrant(X_train_full, DELTA)
X_test, _ = get_upper_quadrant(
    X_test_full,
    DELTA,
    anchor_points=anchor_points
)

print("Anchor points:", anchor_points)
print("Training exceedances:", len(X_train))
print("Test exceedances:", len(X_test))

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(
    X_train_full[:,0], X_train_full[:,1],
    s=5, alpha=0.10, label="Full sample"
)
plt.scatter(
    X_train[:,0], X_train[:,1],
    s=8, alpha=0.5, label="Upper quadrant"
)
plt.axvline(anchor_points[0], linestyle="--")
plt.axhline(anchor_points[1], linestyle="--")
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$X^{(1)}$")
plt.ylabel(r"$X^{(2)}$")
plt.title(
    r"$\mathcal{Q}(\boldsymbol{\delta}_n)$, "
    r"$\boldsymbol{\delta}_n=(0.1,0.1)^\top$"
)
plt.legend()
plt.show()

# 3. ExceedGAN is an extrapolation model

For one margin,

$$
Y=X\mid X>q_X(1-\delta).
$$

If $Z\sim U(0,1)$, then

$$
q_Y(1-z)=q_X(1-\delta z).
$$

When $n\delta z$ is small, $q_X(1-\delta z)$ may be beyond the observed range. Estimating the exceedance generator is therefore an **extrapolation problem**.

# 4. The log-spacing function

Write

$$
U(t)=q_X(1-1/t)=t^\gamma L(t),
$$

where $L$ is slowly varying. Then

$$
q_Y(1-z)=U\left(\frac{1}{\delta z}\right),
$$

and

$$
\log U\left(\frac{1}{\delta z}\right)
-
\log U\left(\frac{1}{\delta}\right)
=
\gamma\log(1/z)
+
\varphi\left(\log(1/z),\log(1/\delta)\right).
$$

Hence

$$
q_Y(1-z)
=
q_X(1-\delta)\,
z^{-\gamma}
\exp\left[
\varphi\left(\log(1/z),\log(1/\delta)\right)
\right].
$$

# 5. Why eLU?

The second-order approximation in the slides uses

$$
\sigma^E(x)
=
\begin{cases}
x,&x\ge 0,\\
e^x-1,&x<0,
\end{cases}
$$

which is PyTorch's `nn.ELU(alpha=1)`.

The public fixed-level implementation can be written schematically as

$$
G_\theta^{\mathrm{EX}}(Z)
=
\widehat u_n
\odot
\exp\left[
\sigma^R
\left\{
N_\theta(-\log Z,-\log\boldsymbol\delta_n)
\right\}
\right].
$$

The empirical threshold is therefore built directly into the generator.

# 6. Construct the three generators

As in the revised EV-GAN notebook, we use a more flexible bounded latent vector and wider networks:

$$
d'=32,\qquad J_G=64,\qquad J_D=32.
$$

The bounded latent domain still leaves the standard GAN with bounded support, but the extra latent dimensions make the adversarial optimisation much less restrictive.

We also split the observed exceedances into a fitting sample and a small validation sample. The validation sample is used only to select a training checkpoint; the independent test sample remains untouched until the final comparison.

In [ ]:
DIM_DATA = 2
LATENT_DIM = 32
HIDDEN_G = 64
HIDDEN_D = (32, 32)

BATCH_SIZE = 32
LR_G = 5e-4
LR_D = 5e-4

N_STEPS = 5000
CHECK_EVERY = 250
SEEDS = [123, 456, 789]

def sample_Z(n):
    return torch.rand(
        n, LATENT_DIM, device=device
    ).clamp(1e-5, 1.0-1e-5)

# Reproducible fit/validation split of the exceedance sample.
rng_split = np.random.default_rng(SEED)
perm = rng_split.permutation(len(X_train))
n_fit = int(0.80 * len(X_train))

X_fit = X_train[perm[:n_fit]]
X_val = X_train[perm[n_fit:]]

print("Exceedances used for fitting:", len(X_fit))
print("Exceedances used for validation:", len(X_val))

## 6.1 Standard GAN

The standard generator uses all 32 latent coordinates. A final `Softplus` keeps the output positive, which is appropriate for Burr data, but the generator is **not** forced to lie above the extreme threshold.

Because the latent cube is compact, the standard GAN still has bounded fitted support.

In [ ]:
class StandardGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_G),
            nn.ReLU(),
            nn.Linear(HIDDEN_G, DIM_DATA)
        )

        nn.init.normal_(self.layers[-1].weight, mean=0.0, std=0.02)

        # Start near the centre of the observed exceedance sample.
        med = np.median(X_fit, axis=0)
        inv_softplus = np.log(np.expm1(np.maximum(med, 1e-3)))
        with torch.no_grad():
            self.layers[-1].bias.copy_(
                torch.tensor(inv_softplus, dtype=torch.float32)
            )

    def forward(self, z):
        return F.softplus(self.layers(z)) + 1e-6

## 6.2 EV-GAN

EV-GAN uses

$$
H_u^{-1}(x)
=
\left(\frac{1-u^2}{2}\right)^{-x}
$$

after adding the corrected-TIF terms. The first two latent coordinates supply the two marginal inverse-TIF transforms, while all 32 coordinates enter the neural network.

As in the revised one-dimensional notebook, the assembled TIF is **not** passed through a final ReLU: the TIF may legitimately be negative away from the far upper tail.

In [ ]:
class TIFInverse(nn.Module):
    def forward(self, u, x):
        u = u.clamp(1e-5, 1.0-1e-5)
        base = ((1.0-u.pow(2))/2.0).clamp_min(1e-8)
        return torch.exp((-x*torch.log(base)).clamp(max=20.0))


class EVGenerator(nn.Module):
    def __init__(self, li_table_size=4096):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(LATENT_DIM, HIDDEN_G),
            nn.ReLU(),
            nn.Linear(HIDDEN_G, DIM_DATA)
        )
        self.tif_inverse = TIFInverse()

        self.var1 = nn.Parameter(torch.full((1,DIM_DATA), 0.25))
        self.var2 = nn.Parameter(torch.zeros(1,DIM_DATA))
        self.var3 = nn.Parameter(torch.zeros(1,DIM_DATA))
        self.var4 = nn.Parameter(torch.zeros(1,DIM_DATA))
        self.var5 = nn.Parameter(torch.zeros(1,DIM_DATA))
        self.var6 = nn.Parameter(torch.zeros(1,DIM_DATA))

        nn.init.normal_(self.layers[-1].weight, mean=0.0, std=0.01)

        # Data-informed initial TIF level at u=0.5.
        med = np.median(X_fit, axis=0)
        tif0 = -np.log(np.maximum(med, 1e-6)) / np.log((1.0-0.5**2)/2.0)
        with torch.no_grad():
            self.layers[-1].bias.copy_(
                torch.tensor(tif0, dtype=torch.float32)
            )

        # Precompute li(1-u) once to avoid a SciPy CPU round-trip
        # during every minibatch.
        self.li_table_size = li_table_size
        self.u_min = 1e-5
        self.u_max = 1.0 - 1e-5

        grid = np.linspace(
            self.u_min,
            self.u_max,
            li_table_size,
            dtype=np.float64
        )
        values = expi(np.log(1.0-grid)).astype(np.float32)

        self.register_buffer(
            "_li_table",
            torch.tensor(values, dtype=torch.float32)
        )

    @staticmethod
    def p01(u):
        return -4.0*u**5 + 5.0*u**4

    @staticmethod
    def p00(u):
        return u**3 - 2.0*u**2 + u

    def li_one_minus_u(self, u):
        pos = (
            (u-self.u_min)
            / (self.u_max-self.u_min)
            * (self.li_table_size-1)
        )

        i0 = torch.floor(pos).long().clamp(0, self.li_table_size-2)
        w = pos - i0.to(pos.dtype)

        v0 = self._li_table[i0]
        v1 = self._li_table[i0+1]

        return v0 + (v1-v0)*w

    def Phi(self, u):
        v = (1.0-u).clamp_min(1e-6)
        logv = torch.log(v)
        liv = self.li_one_minus_u(u)

        return (
            self.var2/logv
            + self.var3*liv
            + self.var4*(v/logv-liv)
            + self.var5*(v*(logv+1.0)/(2.0*logv**2)-liv/2.0)
        )

    def corrected_tif(self, z):
        z = z.clamp(1e-5, 1.0-1e-5)
        u = z[:,:DIM_DATA]

        network = self.layers(z)

        corr1 = self.p01(u)*(self.var1+self.Phi(u))
        corr1 = torch.nan_to_num(
            corr1, nan=0.0, posinf=0.0, neginf=0.0
        )
        corr2 = self.var6*self.p00(u)

        return network + corr1 + corr2

    def forward(self, z):
        z = z.clamp(1e-5, 1.0-1e-5)
        u = z[:,:DIM_DATA]
        f_tif = self.corrected_tif(z)
        return self.tif_inverse(u, f_tif)

## 6.3 Fixed-Level ExceedGAN

The fixed-level generator conditions on $-\log \boldsymbol{\delta}_n$, uses eLU hidden units, and multiplies the final extrapolation factor by the empirical anchor point.

The output therefore satisfies

$$
G_\theta^{\mathrm{EX}}(Z)\geq \widehat u_n
$$

componentwise by construction.

We initialise the final bias using the median empirical log-ratio above the anchor. This gives the generator a sensible starting scale before adversarial training begins.

In [ ]:
class ExceedGenerator(nn.Module):
    def __init__(self, anchor_levels, anchor_points):
        super().__init__()

        self.register_buffer(
            "anchor_levels",
            torch.tensor(
                anchor_levels, dtype=torch.float32
            ).reshape(1,-1)
        )
        self.register_buffer(
            "anchor_points",
            torch.tensor(
                anchor_points, dtype=torch.float32
            ).reshape(1,-1)
        )

        self.layers = nn.Sequential(
            nn.Linear(LATENT_DIM + DIM_DATA, HIDDEN_G),
            nn.ELU(alpha=1.0),
            nn.Linear(HIDDEN_G, DIM_DATA)
        )

        nn.init.normal_(
            self.layers[-1].weight,
            mean=0.0,
            std=0.01
        )

        log_ratio_med = np.log(
            np.median(
                X_fit / anchor_points.reshape(1,-1),
                axis=0
            )
        )
        log_ratio_med = np.clip(log_ratio_med, 0.02, 2.0)

        with torch.no_grad():
            self.layers[-1].bias.copy_(
                torch.tensor(log_ratio_med, dtype=torch.float32)
            )

    def forward(self, z):
        delta_batch = self.anchor_levels.expand(z.shape[0], -1)

        transformed = -torch.log(
            torch.cat([z, delta_batch], dim=1).clamp_min(1e-6)
        )

        network = self.layers(transformed)

        # Theoretical FL-ExceedGAN form: anchor times exp(ReLU(network)).
        # A moderate upper clamp only prevents numerical overflow.
        return self.anchor_points * torch.exp(
            F.relu(network).clamp(max=8.0)
        )

# 7. Common discriminator and adversarial loss

All three models use the same discriminator and the same non-saturating GAN loss.

A major numerical change is that the discriminator no longer sees the raw heavy-tailed observations. Instead it sees standardised **log-ratios relative to the threshold**,

$$
R^{(m)}
=
\frac{
\log\{X^{(m)}/\widehat u_n^{(m)}\}-a_m
}{b_m}.
$$

This is a one-to-one transformation on the positive support, so it does not change the distribution-learning problem, but it prevents a few very large observations from dominating the discriminator.

In [ ]:
log_ratio_fit = np.log(
    np.maximum(X_fit, 1e-8)
    / anchor_points.reshape(1,-1)
)

DISC_MEAN = log_ratio_fit.mean(axis=0).astype(np.float32)
DISC_SD = log_ratio_fit.std(axis=0).astype(np.float32)
DISC_SD = np.maximum(DISC_SD, 1e-3)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.register_buffer(
            "anchor",
            torch.tensor(
                anchor_points,
                dtype=torch.float32
            ).reshape(1,-1)
        )
        self.register_buffer(
            "disc_mean",
            torch.tensor(
                DISC_MEAN,
                dtype=torch.float32
            ).reshape(1,-1)
        )
        self.register_buffer(
            "disc_sd",
            torch.tensor(
                DISC_SD,
                dtype=torch.float32
            ).reshape(1,-1)
        )

        self.layers = nn.Sequential(
            nn.Linear(DIM_DATA, HIDDEN_D[0]),
            nn.ReLU(),
            nn.Linear(HIDDEN_D[0], HIDDEN_D[1]),
            nn.ReLU(),
            nn.Linear(HIDDEN_D[1], 1)
        )

    def transform(self, x):
        x = x.clamp_min(1e-8)
        r = torch.log(x / self.anchor)
        return (r-self.disc_mean)/self.disc_sd

    def forward(self, x):
        return self.layers(self.transform(x)).squeeze(-1)

# 8. Train all three models on the same exceedance sample

The earlier notebook used one seed, 120 epochs and the final training iterate. That can give a misleading comparison because GAN training is stochastic and the final iterate need not be the best one.

The revised setup uses:

- batch size 32;
- learning rates $5\times10^{-4}$;
- 5000 minibatch updates;
- no gradient clipping;
- three random seeds;
- checkpoint selection using the held-out validation exceedances.

The checkpoint score combines marginal upper-tail accuracy with a small dependence penalty.

In [ ]:
def marginal_tail_msle(reference, generated, xi=0.90):
    reference = np.asarray(reference)
    generated = np.asarray(generated)

    out = []

    for m in range(DIM_DATA):
        x_ref = np.sort(reference[:,m])
        x_gen = np.sort(generated[:,m])

        k = max(
            5,
            int(np.ceil(
                (1.0-xi)*min(len(x_ref), len(x_gen))
            ))
        )

        q_ref = np.maximum(x_ref[-k:], 1e-12)
        q_gen = np.maximum(x_gen[-k:], 1e-12)

        out.append(
            np.mean(
                (np.log(q_ref)-np.log(q_gen))**2
            )
        )

    return np.asarray(out)


def validation_score(reference, generated, xi=0.90):
    if len(generated) < 100:
        return np.inf

    marginal = marginal_tail_msle(
        reference,
        generated,
        xi=xi
    ).mean()

    tau_ref = kendalltau(
        reference[:,0], reference[:,1]
    ).statistic
    tau_gen = kendalltau(
        generated[:,0], generated[:,1]
    ).statistic

    dep_penalty = (tau_gen-tau_ref)**2

    return float(marginal + dep_penalty)


@torch.no_grad()
def generate_for_validation(
    generator,
    n_target,
    by_construction=False,
    proposal_factor=12
):
    generator.eval()

    if by_construction:
        return generator(
            sample_Z(n_target)
        ).cpu().numpy()

    # Standard GAN and EV-GAN are not constrained above the threshold.
    # Draw a moderately large candidate pool and retain the upper quadrant.
    n_prop = max(
        n_target*proposal_factor,
        5000
    )

    candidate = generator(
        sample_Z(n_prop)
    ).cpu().numpy()

    keep = np.all(
        candidate
        > anchor_points.reshape(1,-1),
        axis=1
    )

    accepted = candidate[keep]

    if len(accepted) > n_target:
        accepted = accepted[:n_target]

    return accepted


def train_model(
    generator,
    X_real,
    X_validation,
    by_construction=False,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    lr_G=LR_G,
    lr_D=LR_D,
    seed=SEED,
    check_every=CHECK_EVERY
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    generator = generator.to(device)
    discriminator = Discriminator().to(device)

    opt_G = torch.optim.Adam(
        generator.parameters(),
        lr=lr_G
    )
    opt_D = torch.optim.Adam(
        discriminator.parameters(),
        lr=lr_D
    )

    criterion = nn.BCEWithLogitsLoss()

    X_tensor = torch.tensor(
        X_real,
        dtype=torch.float32,
        device=device
    )

    history_G = []
    history_D = []
    history_step = []

    block_G = []
    block_D = []

    best_score = np.inf
    best_step = None
    best_state = None

    for step in range(1, n_steps+1):

        idx = torch.randint(
            0,
            X_tensor.shape[0],
            (batch_size,),
            device=device
        )
        x_real = X_tensor[idx]

        # 1. Discriminator update
        z = sample_Z(batch_size)

        with torch.no_grad():
            x_fake = generator(z)

        real_logits = discriminator(x_real)
        fake_logits = discriminator(x_fake)

        loss_D = (
            criterion(
                real_logits,
                torch.ones_like(real_logits)
            )
            +
            criterion(
                fake_logits,
                torch.zeros_like(fake_logits)
            )
        )

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # 2. Generator update
        z = sample_Z(batch_size)
        x_fake = generator(z)
        fake_logits = discriminator(x_fake)

        loss_G = criterion(
            fake_logits,
            torch.ones_like(fake_logits)
        )

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        block_D.append(loss_D.item())
        block_G.append(loss_G.item())

        if step % check_every == 0:

            history_step.append(step)
            history_D.append(np.mean(block_D))
            history_G.append(np.mean(block_G))

            block_D = []
            block_G = []

            generated_val = generate_for_validation(
                generator,
                n_target=min(
                    2000,
                    len(X_validation)
                ),
                by_construction=by_construction
            )

            score = validation_score(
                X_validation[:len(generated_val)],
                generated_val
            )

            if np.isfinite(score) and score < best_score:
                best_score = score
                best_step = step
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value
                    in generator.state_dict().items()
                }

            if step % (4*check_every) == 0:
                print(
                    f"step {step:5d}/{n_steps} | "
                    f"D={history_D[-1]:.3f} | "
                    f"G={history_G[-1]:.3f} | "
                    f"best validation={best_score:.4f}"
                )

    if best_state is not None:
        generator.load_state_dict(best_state)

    return {
        "G": generator,
        "D": discriminator,
        "hist_G": history_G,
        "hist_D": history_D,
        "hist_step": history_step,
        "score": best_score,
        "best_step": best_step,
        "seed": seed
    }


def fit_seed_ensemble(
    generator_factory,
    model_name,
    by_construction=False
):
    runs = []

    for seed in SEEDS:

        print(f"\n{model_name}: seed {seed}")

        torch.manual_seed(seed)
        generator = generator_factory()

        run = train_model(
            generator,
            X_fit,
            X_val,
            by_construction=by_construction,
            seed=seed
        )

        runs.append(run)

        print(
            f"selected checkpoint: "
            f"step {run['best_step']}; "
            f"validation score={run['score']:.4f}"
        )

    finite_runs = [
        r for r in runs
        if np.isfinite(r["score"])
    ]

    if len(finite_runs) == 0:
        raise RuntimeError(
            f"No usable {model_name} run was found."
        )

    finite_runs = sorted(
        finite_runs,
        key=lambda r: r["score"]
    )

    chosen = finite_runs[
        len(finite_runs)//2
    ]

    print(f"\n{model_name} seed summary")
    for r in finite_runs:
        print(
            f"seed={r['seed']:3d} | "
            f"score={r['score']:.4f} | "
            f"checkpoint={r['best_step']}"
        )

    print(
        f"Using median-score run: "
        f"seed {chosen['seed']}"
    )

    return chosen, runs

## 8.1 Standard GAN

The default fits three seeds and retains the median validation-score run.

For a faster live demonstration, temporarily set

```python
SEEDS = [123]
N_STEPS = 3000
```

in the settings cell.

In [ ]:
chosen_gan, runs_gan = fit_seed_ensemble(
    lambda: StandardGenerator(),
    "Standard GAN",
    by_construction=False
)

G_gan = chosen_gan["G"]
D_gan = chosen_gan["D"]
hist_G_gan = chosen_gan["hist_G"]
hist_D_gan = chosen_gan["hist_D"]
hist_step_gan = chosen_gan["hist_step"]

## 8.2 EV-GAN

EV-GAN uses the same training schedule, discriminator, validation rule and seed set. Only the generator parametrisation changes.

In [ ]:
chosen_ev, runs_ev = fit_seed_ensemble(
    lambda: EVGenerator(),
    "EV-GAN",
    by_construction=False
)

G_ev = chosen_ev["G"]
D_ev = chosen_ev["D"]
hist_G_ev = chosen_ev["hist_G"]
hist_D_ev = chosen_ev["hist_D"]
hist_step_ev = chosen_ev["hist_step"]

## 8.3 Fixed-Level ExceedGAN

FL-ExceedGAN is generated directly above the estimated anchor, so its validation simulation does not need rejection.

In [ ]:
chosen_ex, runs_ex = fit_seed_ensemble(
    lambda: ExceedGenerator(
        DELTA,
        anchor_points
    ),
    "FL-ExceedGAN",
    by_construction=True
)

G_ex = chosen_ex["G"]
D_ex = chosen_ex["D"]
hist_G_ex = chosen_ex["hist_G"]
hist_D_ex = chosen_ex["hist_D"]
hist_step_ex = chosen_ex["hist_step"]

# 9. Training losses

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))

axes[0].plot(
    hist_step_gan, hist_D_gan,
    label="GAN"
)
axes[0].plot(
    hist_step_ev, hist_D_ev,
    label="EV-GAN"
)
axes[0].plot(
    hist_step_ex, hist_D_ex,
    label="FL-ExceedGAN"
)
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Discriminator loss")
axes[0].legend()

axes[1].plot(
    hist_step_gan, hist_G_gan,
    label="GAN"
)
axes[1].plot(
    hist_step_ev, hist_G_ev,
    label="EV-GAN"
)
axes[1].plot(
    hist_step_ex, hist_G_ex,
    label="FL-ExceedGAN"
)
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Generator loss")
axes[1].legend()

plt.tight_layout()
plt.show()

# 10. Simulate upper-quadrant observations

For the standard GAN and EV-GAN we use **acceptance-rejection**: generate candidates and retain those above both anchor points.

FL-ExceedGAN generates above the anchor by construction, so no rejection step is needed.

In [ ]:
@torch.no_grad()
def simulate_upper_quadrant(
    generator,
    n,
    anchor_points,
    by_construction=False,
    batch_size=8192,
    max_draws=2_000_000
):
    generator.eval()

    if by_construction:
        return (
            generator(
                sample_Z(n)
            ).cpu().numpy(),
            1.0
        )

    anchors = np.asarray(
        anchor_points,
        dtype=np.float32
    ).reshape(1,-1)

    accepted = []
    n_accepted = 0
    n_drawn = 0

    while n_accepted < n:

        if n_drawn >= max_draws:
            raise RuntimeError(
                "Maximum proposal count reached. "
                "The fitted model is generating too few "
                "upper-quadrant observations."
            )

        m = min(
            batch_size,
            max_draws-n_drawn
        )

        candidate = generator(
            sample_Z(m)
        ).cpu().numpy()

        n_drawn += m

        keep = np.all(
            candidate > anchors,
            axis=1
        )

        if np.any(keep):
            new = candidate[keep]
            accepted.append(new)
            n_accepted += len(new)

    out = np.concatenate(
        accepted,
        axis=0
    )[:n]

    return out, n/n_drawn


N_COMPARE = min(10_000, len(X_test))
X_ref = X_test[:N_COMPARE]

X_gan, rate_gan = simulate_upper_quadrant(
    G_gan,
    N_COMPARE,
    anchor_points,
    by_construction=False
)

X_ev, rate_ev = simulate_upper_quadrant(
    G_ev,
    N_COMPARE,
    anchor_points,
    by_construction=False
)

X_ex, rate_ex = simulate_upper_quadrant(
    G_ex,
    N_COMPARE,
    anchor_points,
    by_construction=True
)

print(f"GAN acceptance rate:    {rate_gan:.3f}")
print(f"EV-GAN acceptance rate: {rate_ev:.3f}")
print(f"FL-ExceedGAN rate:      {rate_ex:.3f}")

# 11. Visual comparison

As in the slides, plot the true test exceedances as black crosses and the generated samples on log-log axes.

In [ ]:
fig, axes = plt.subplots(
    1,3,
    figsize=(16,5),
    sharex=True,
    sharey=True
)

for ax, (name, X_sim) in zip(
    axes,
    [
        ("GAN", X_gan),
        ("EV-GAN", X_ev),
        ("FL-ExceedGAN", X_ex)
    ]
):
    ax.scatter(
        X_ref[:,0], X_ref[:,1],
        s=8, alpha=0.18,
        marker="x",
        label="True test"
    )
    ax.scatter(
        X_sim[:,0], X_sim[:,1],
        s=8, alpha=0.28,
        label=name
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$X^{(1)}$")
    ax.set_title(name)

axes[0].set_ylabel(r"$X^{(2)}$")
axes[0].legend()

plt.suptitle(
    r"$\mu=2,\ \gamma=0.5,\ "
    r"(\rho_1,\rho_2)=(-3,-3),\ "
    r"\boldsymbol{\delta}_n=(0.1,0.1)^\top$"
)

plt.tight_layout()
plt.show()

# 12. Marginal tail comparison

The earlier notebook averaged log-quantile error from $p=0.5$, which can make a model look good even when its most extreme conditional quantiles are poor.

Here the default diagnostic focuses on

$$
p\in[0.90,0.999].
$$

This is more aligned with the scientific purpose of ExceedGAN.

In [ ]:
p_grid = np.linspace(0.90, 0.999, 150)

def marginal_log_quantile_rmse(reference, generated):
    out = []

    for m in range(DIM_DATA):
        q_ref = np.quantile(
            reference[:,m],
            p_grid
        )
        q_gen = np.quantile(
            generated[:,m],
            p_grid
        )

        out.append(
            np.sqrt(
                np.mean(
                    (
                        np.log(q_gen)
                        - np.log(q_ref)
                    )**2
                )
            )
        )

    return np.asarray(out)

errors = {}

for name, X_sim in [
    ("GAN", X_gan),
    ("EV-GAN", X_ev),
    ("FL-ExceedGAN", X_ex)
]:
    errors[name] = marginal_log_quantile_rmse(
        X_ref,
        X_sim
    )

pd.DataFrame({
    "Model": list(errors.keys()),
    "Margin 1": [
        errors[k][0] for k in errors
    ],
    "Margin 2": [
        errors[k][1] for k in errors
    ],
    "Mean": [
        errors[k].mean() for k in errors
    ]
}).sort_values("Mean")

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))

for m, ax in enumerate(axes):
    x = -np.log(1-p_grid)

    ax.plot(
        x, np.log(np.quantile(X_ref[:,m], p_grid)),
        label="True test", linewidth=2
    )
    ax.plot(
        x, np.log(np.quantile(X_gan[:,m], p_grid)),
        label="GAN"
    )
    ax.plot(
        x, np.log(np.quantile(X_ev[:,m], p_grid)),
        label="EV-GAN"
    )
    ax.plot(
        x, np.log(np.quantile(X_ex[:,m], p_grid)),
        label="FL-ExceedGAN"
    )

    ax.set_xlabel(r"$-\log(1-p)$")
    ax.set_ylabel(r"$\log q_m(p)$")
    ax.set_title(f"Margin {m+1}")

axes[0].legend()
plt.tight_layout()
plt.show()

# 13. Dependence comparison

The ExceedGAN slides focus on marginal tail accuracy, but for a bivariate workshop it is useful to add a dependence diagnostic.

We compare Kendall's $\tau$ of the generated upper-quadrant samples with the independent test exceedances. This is a **post-training diagnostic**, not part of the GAN loss.

In [ ]:
def sample_tau(X):
    return kendalltau(X[:,0], X[:,1]).statistic

tau_ref = sample_tau(X_ref)

dep = pd.DataFrame({
    "Model": ["True test","GAN","EV-GAN","FL-ExceedGAN"],
    "Kendall tau": [
        tau_ref,
        sample_tau(X_gan),
        sample_tau(X_ev),
        sample_tau(X_ex)
    ]
})
dep["Absolute error"] = np.abs(dep["Kendall tau"]-tau_ref)
dep

# 14. Extreme conditional quantiles

Compare a few high quantiles **within the exceedance distribution**.

In [ ]:
rows = []

for m in range(DIM_DATA):
    for p in [0.90,0.95,0.99,0.995]:
        rows.append({
            "Margin": m+1,
            "p": p,
            "True test": np.quantile(X_ref[:,m], p),
            "GAN": np.quantile(X_gan[:,m], p),
            "EV-GAN": np.quantile(X_ev[:,m], p),
            "FL-ExceedGAN": np.quantile(X_ex[:,m], p)
        })

pd.DataFrame(rows)

# 15. What changes between the models?

### Standard GAN

$$
Z
\rightarrow
\text{ReLU network}
\rightarrow
\operatorname{softplus}
\rightarrow
\widetilde X.
$$

It is a generic positive distribution learner, but bounded latent input still implies bounded fitted output.

### EV-GAN

$$
Z
\rightarrow
\widehat f_\theta^{\mathrm{TIF}}
\rightarrow
H_Z^{-1}
\rightarrow
\widetilde X.
$$

Its marginal parametrisation explicitly permits heavy tails.

### Fixed-Level ExceedGAN

$$
(Z,\boldsymbol{\delta}_n)
\rightarrow
-\log(Z,\boldsymbol{\delta}_n)
\rightarrow
\text{eLU network}
\rightarrow
\widehat u_n
\odot
\exp\{\sigma^R(\cdot)\}.
$$

It models the **conditional exceedance distribution itself**, is anchored at the high threshold by construction, and uses an architecture motivated by extreme-quantile extrapolation.

For all three methods, the revised training procedure uses the same log-ratio discriminator, repeated seeds and validation-based checkpoint selection.

# 16. Workshop exercises

### A. Restore the asymmetric slide setting

Set

```python
RHOS = (-1.0, -3.0)
```

and regenerate the data. The first margin is now in a less smooth second-order regime. Which method deteriorates most?

### B. Deeper threshold

Set

```python
DELTA = np.array([0.05, 0.05])
```

and repeat the experiment.

### C. Heavier tail

Set

```python
GAMMA = 0.9
```

and regenerate the data.

### D. Change extremal dependence

Try

```python
MU = 1.1
```

and

```python
MU = 10
```

matching the range of Gumbel dependence parameters in the slides.

### E. Inspect seed variability

Compare

```python
runs_gan
runs_ev
runs_ex
```

and inspect the selected validation scores and checkpoint locations.

### F. Remove eLU

Replace `nn.ELU(alpha=1.0)` in `ExceedGenerator` by `nn.ReLU()`. Does tail extrapolation worsen?

# References

Allouche, M., Girard, S. and Gobet, E. (2022). **EV-GAN: Simulation of extreme events with ReLU neural networks.** *Journal of Machine Learning Research*, 23.

Allouche, M., Girard, S. and Gobet, E. (2026). **ExceedGAN: Simulation above extreme thresholds using Generative Adversarial Networks.** *Extremes*.

The mathematical progression follows pp. 19–28 of the supplied slides. The computational forms of the GAN, EV-GAN and fixed-level ExceedGAN generators are aligned with the authors' public PyTorch implementation.